# Chapter 2: Thinking Like an Analyst

Companion notebook for *Master Exploratory Data Analysis: A Hands-On Beginner's Guide with Python, Statistics & A/B Testing*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files are in the same folder as this notebook.

## 2.1 The EDA Workflow

### Step one: understand what you have

In [1]:
import pandas as pd

names = ['customers', 'orders', 'plans', 'cancellations',
         'support_tickets', 'survey', 'ab_test']
tables = {}
for name in names:
    tables[name] = pd.read_csv(f'{name}.csv')
    rows, cols = tables[name].shape
    print(f'{name:16}{rows:>7,} rows {cols:>3} columns')

customers         4,210 rows   9 columns
orders           28,772 rows   8 columns
plans                 3 rows   3 columns
cancellations     2,369 rows   3 columns
support_tickets   1,380 rows   5 columns
survey              532 rows   6 columns
ab_test          22,756 rows   8 columns


## 2.2 Kinds of Variables

In [2]:
customers = pd.read_csv('customers.csv', parse_dates=['signup_date'])
orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
customers.dtypes

customer_id             int64
signup_date    datetime64[us]
city                      str
state                     str
region                    str
age                   float64
channel                   str
plan                      str
device                    str
dtype: object

### Ordinal variables

In [3]:
survey = pd.read_csv('survey.csv', parse_dates=['response_date'])
low = survey['satisfaction'].min()
high = survey['satisfaction'].max()
print(f'satisfaction runs from {low} to {high}')

satisfaction runs from 1 to 5


In [4]:
pd.read_csv('plans.csv')

,plan,monthly_price,bags_per_month
0,Basic,16.0,1
1,Standard,28.0,2
2,Premium,49.0,4


### Date variables

In [5]:
first = customers['signup_date'].min().date()
last = customers['signup_date'].max().date()
print(f'Sign-ups run from {first} to {last}')

Sign-ups run from 2024-01-01 to 2025-12-31


### IDs that look like numbers

In [6]:
ids = customers['customer_id']
print(f'Average customer_id: {ids.mean()}')
print(f'Distinct IDs: {ids.nunique()} for {len(ids)} customers')

Average customer_id: 3105.5
Distinct IDs: 4210 for 4210 customers


## 2.3 Asking Questions You Can Answer

### `s.isin()`

In [7]:
cancellations = pd.read_csv('cancellations.csv',
                            parse_dates=['cancel_date'])
customers['cancelled'] = customers['customer_id'].isin(
    cancellations['customer_id'])
customers[['customer_id', 'plan', 'cancelled']].head()

,customer_id,plan,cancelled
0,1001,Basic,False
1,1002,Basic,True
2,1003,Standard,True
3,1004,Basic,True
4,1005,Basic,True


In [8]:
flagged = customers['cancelled'].sum()
print(f'Customers flagged: {flagged:,}')
print(f'Rows in cancellations: {len(cancellations):,}')

Customers flagged: 2,369
Rows in cancellations: 2,369


## 2.4 Building a Data Dictionary

### `df.isna().sum()`

In [9]:
customers.isna().sum()

customer_id      0
signup_date      0
city             0
state            0
region           0
age            167
channel          0
plan             0
device           0
cancelled        0
dtype: int64

### A function that builds the dictionary

In [10]:
def data_dictionary(table, kinds):
    return pd.DataFrame({
        'dtype': table.dtypes.astype(str),
        'kind': pd.Series(kinds),
        'missing': table.isna().sum(),
        'unique': table.nunique(),
        'example': table.iloc[1002],
    })

customer_kinds = {
    'customer_id': 'ID', 'signup_date': 'date', 'city': 'category',
    'state': 'category', 'region': 'category', 'age': 'numeric',
    'channel': 'category', 'plan': 'ordinal', 'device': 'category',
    'cancelled': 'flag',
}
data_dictionary(customers, customer_kinds)

,dtype,kind,missing,unique,example
customer_id,int64,ID,0,4210,2003
signup_date,datetime64[us],date,0,725,2024-08-29 00:00:00
city,str,category,0,20,Los Angeles
state,str,category,0,17,CA
region,str,category,0,4,West
age,float64,numeric,167,56,46.0
channel,str,category,0,5,Email
plan,str,ordinal,0,3,Standard
device,str,category,0,2,Desktop
cancelled,bool,flag,0,2,True


In [11]:
order_kinds = {
    'order_id': 'ID', 'customer_id': 'ID', 'order_date': 'date',
    'order_type': 'category', 'items': 'numeric',
    'amount': 'numeric', 'discount': 'numeric',
    'delivery_days': 'numeric',
}
data_dictionary(orders, order_kinds)

,dtype,kind,missing,unique,example
order_id,int64,ID,0,28772,101003
customer_id,int64,ID,0,4210,1206
order_date,datetime64[us],date,0,731,2024-04-19 00:00:00
order_type,str,category,0,2,Subscription
items,int64,numeric,0,18,1
amount,float64,numeric,0,2712,28.0
discount,float64,numeric,0,4,0.0
delivery_days,int64,numeric,0,13,4
